# Silice: control longitudinal D16-C1

CPU estándar, un hilo. Protocolo preregistrado, sin resultado óptico todavía. Dos familias a32768pasos, 2mm, matriz Duffy16. Preservar todos los dictámenes. No generar malla fina ni avanzar tareas. La transferencia y ejecución remota requieren aprobación específica.

In [ ]:
import os, sys, json, platform, subprocess, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone
print({'utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'platform':platform.system()})

Preparación aislada: dependencias fijadas en almacenamiento temporal. No montar Drive ni contratar recursos.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--target','/content/silice_d16_deps','numpy==2.2.6','scipy==1.15.1','psutil==6.1.1'],check=True)

In [ ]:
from google.colab import files
uploaded=files.upload()
assert set(uploaded)=={'Silice_D16_C1_20261008.zip'}
archive=Path('Silice_D16_C1_20261008.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='cebaeac0f2952360f319acd9081f0f6149e0fbda7112258ac738aafc6d0a3ea1'
folder=Path('/content/silice_d16_c1_20261008')
assert not folder.exists()
folder.mkdir()
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in z.namelist())
    z.extractall(folder)
project=folder/'project'
manifest=json.loads((folder/'transfer_manifest.json').read_text())
for row in manifest['files']:
    p=project/row['path']
    assert p.stat().st_size==row['bytes'] and hashlib.sha256(p.read_bytes()).hexdigest()==row['sha256']
print({'input_integrity_pass':True,'files':len(manifest['files'])})

Ejecutar una vez. Un fallo se conserva y detiene esta etapa. No continuar por un dictamen negativo sin revisarlo. Tiempo máximo22000s, recursos limitados por contrato; mantener el uso interactivo normal de Colab.

In [ ]:
environment=os.environ.copy()
environment['PYTHONPATH']='/content/silice_d16_deps'
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    environment[key]='1'
out=project/'resultados/codex/cloud_d16_c1_20261008'
child=subprocess.run([sys.executable,str(project/'scripts/run_point03_d16_time.py'),'--out',str(out)],env=environment,timeout=22030)
print({'returncode':child.returncode})
execution=json.loads((out/'execution.json').read_text())
print(json.dumps(execution,indent=2))
if execution['status']=='completed':
    subprocess.run([sys.executable,str(project/'scripts/audit_point03_d16_time.py'),'--out',str(out)],env=environment,check=True,timeout=600)
    print((out/'integrity_audit.json').read_text())

Descargar evidencia incluso si el ensayo falla. Verificar hashes localmente antes de adoptar resultados. Un aprobado temporal no cierra T96/Q4.

In [ ]:
result=Path('/content/Silice_D16_C1_resultados_20261008.zip')
assert not result.exists()
with zipfile.ZipFile(result,'x',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in out.rglob('*'):
        if p.is_file():
            z.write(p,p.relative_to(project).as_posix())
    z.writestr('runtime.json',json.dumps({'python':sys.version,'platform':platform.system(),'utc':datetime.now(timezone.utc).isoformat()},indent=2))
print({'result_sha256':hashlib.sha256(result.read_bytes()).hexdigest()})
files.download(str(result))